# SI4006 · M4 · Amparo — **El sistema ya ve** · Pipeline multimodal

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 4 · Entrega M4

---

Patrón **input**: el usuario adjunta la **foto de un documento jurídico** y un VLM
(**Qwen2-VL-2B**) lo lee y extrae **hechos en JSON**. Esos hechos se concatenan al
relato y entran al sistema de M3 de Amparo, que decide el mecanismo con su motor de
reglas. **M3 no se reescribe**: `ver()` es un traductor en la puerta.

> El VLM **extrae hechos del documento, no decide derecho** — es el mismo principio
> de Amparo ("el LLM extrae, las reglas deciden"), ahora con una imagen en la entrada.

**Estado de este notebook:**
- **T1 · Santiago** — setup reproducible + `cargar()` + patrón + imágenes de ejemplo. ✅
- **T2 · Santiago** — la función `ver()` con Qwen2-VL-2B + válvula + JSON del dominio. ✅
- **T3–T4 · Lucas** — imágenes sintéticas + `casos_m4.csv`. (pendiente)
- **T5–T6 · Nico** — harness + scorecard + tabla de delta. (pendiente)
- **T7 · Martín/Tomás** — reemplazar el **sustituto** por el M3 real. (pendiente)

Corre en Colab con GPU **T4**. No ejecutar en local sin GPU.

## 0 · Setup (T1)

Debe correr desde cero en un Colab limpio: **Entorno → Reiniciar y ejecutar todo**.

In [ ]:
%pip install -q "transformers>=4.45" accelerate pillow qwen-vl-utils pandas
print('Listo.')

In [ ]:
import torch, requests, io, json, re, time
import pandas as pd
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device != 'cuda':
    print('AVISO: sin GPU el VLM va lentisimo. En Colab: Entorno de ejecucion -> Cambiar tipo -> T4 GPU.')

def cargar(src):
    """Carga una imagen desde URL o ruta local y la normaliza a RGB."""
    if str(src).startswith('http'):
        return Image.open(io.BytesIO(requests.get(src, timeout=30).content)).convert('RGB')
    return Image.open(src).convert('RGB')

## 1 · El patrón y las imágenes (T1)

Decisión ya tomada en el plan (`m4/README_PLAN.md`): patrón **input**. La pregunta del
usuario que necesita ojos es: *"adjunto este documento, ¿qué mecanismo constitucional
me corresponde?"*.

Las imágenes de ejemplo (COCO) son solo para que el notebook corra de punta a punta hoy.
Lucas (T3) las reemplaza por los **documentos jurídicos sintéticos** de
`m4/eval_set/imagenes/`.

In [ ]:
PATRON = 'input'   # M4 de Amparo: el usuario manda una foto de un documento, el VLM la lee

# La pregunta del usuario que necesita ojos (una frase):
PREGUNTA_USUARIO = 'Adjunto este documento que me llegó, ¿qué mecanismo constitucional me corresponde?'

# Imagenes de EJEMPLO (COCO) para que el pipeline corra hoy. Lucas (T3) pone las reales:
#   IMAGENES = {'notificacion_01': 'm4/eval_set/imagenes/notificacion_01.png', ...}
IMAGENES = {
    'ejemplo_texto': 'http://images.cocodataset.org/val2017/000000039769.jpg',
    'ejemplo_sala':  'http://images.cocodataset.org/val2017/000000000139.jpg',
    'ejemplo_esqui': 'http://images.cocodataset.org/val2017/000000000785.jpg',
}
imgs = {k: cargar(v) for k, v in IMAGENES.items()}
assert PATRON == 'input', 'Para Amparo el patron es input (ver README_PLAN.md).'
print('patron:', PATRON, '· imagenes:', list(imgs))

## 2 · La función `ver()` (T2)

Una función, una responsabilidad: **imagen → (texto_crudo, JSON)**. Lo que sale de aquí
entra al sistema de M3 como texto.

Las **tres reglas del patrón input** (S13):
1. **Válvula de escape**: si un dato no se aprecia, el VLM lo deja en `null` en vez de
   inventarlo. Un VLM pequeño alucina fechas y números (S11).
2. **Salida JSON validada**: se pide JSON con campos fijos; si no parsea, el caso
   cuenta como fallo (no se disimula).
3. (Opcional) *describir antes de preguntar*: reduce alucinación pero cuesta el doble.
   Aquí usamos una sola llamada; si hace falta, se mide el delta de latencia.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

VLM_ID = 'Qwen/Qwen2-VL-2B-Instruct'
vlm = Qwen2VLForConditionalGeneration.from_pretrained(
    VLM_ID,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map='auto')
vproc = AutoProcessor.from_pretrained(VLM_ID)
print('VLM listo:', VLM_ID)

@torch.no_grad()
def preguntar(imagen, pregunta, system=None, max_new_tokens=256):
    """Una llamada al VLM: (imagen, pregunta) -> texto. `system` lleva la valvula."""
    msgs = ([{'role': 'system', 'content': system}] if system else []) + \
           [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': pregunta}]}]
    texto = vproc.apply_chat_template(msgs, add_generation_prompt=True)
    ent = vproc(text=[texto], images=[imagen], return_tensors='pt').to(vlm.device)
    out = vlm.generate(**ent, max_new_tokens=max_new_tokens, do_sample=False)
    return vproc.batch_decode(out[:, ent['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()

In [ ]:
# Regla 1: la valvula de escape va en el system prompt.
VALVULA = ('Responde SOLO con lo que se ve en la imagen. Si un dato no se aprecia, '
           'ponlo en null. No inventes fechas, números, entidades ni plazos. '
           'No des opiniones jurídicas: solo describe lo que el documento dice.')

# Instruccion del dominio de Amparo: que extraer del documento y en que formato.
# Son HECHOS legibles del documento, nunca una conclusion juridica.
INSTRUCCION_DOMINIO = (
    'Eres un asistente que lee documentos jurídicos colombianos. Mira la imagen del '
    'documento y devuelve SOLO un JSON (sin texto antes ni después) con estos campos:\n'
    '{\n'
    '  "tipo_documento": uno de "notificacion" | "oficio" | "respuesta_peticion" | "sentencia" | "otro",\n'
    '  "entidad": quién emite el documento (texto corto) o null,\n'
    '  "fecha": la fecha visible en el documento (texto) o null,\n'
    '  "plazo_dias": el número de días de plazo que menciona el documento (entero) o null,\n'
    '  "pretension_visible": qué se le exige o responde al ciudadano, en una frase, o null\n'
    '}\n'
    'Si un campo no se aprecia en la imagen, su valor es null.')

def parsear_json(t):
    """Regla 2: extrae el primer objeto JSON del texto. None si no hay o no parsea."""
    i, j = t.find('{'), t.rfind('}')
    if i == -1 or j == -1:
        return None
    try:
        return json.loads(t[i:j+1])
    except Exception:
        return None

def ver(imagen, instruccion=None):
    """imagen -> (texto_crudo, json_o_None).

    Lo que sale de aqui entra al sistema de M3 como texto (ver celda 3,
    sistema_multimodal). El JSON son hechos del documento; NO es una decision
    juridica: esa la toma el motor de reglas de Amparo.
    """
    salida = preguntar(imagen, instruccion or INSTRUCCION_DOMINIO, system=VALVULA)
    return salida, parsear_json(salida)

### 2D · Probar `ver()` sola, sobre 3 imágenes, con latencia (T2)

Antes de enchufar nada: ¿hace lo que promete? ¿Cuánto tarda? La latencia va al scorecard
(Nico, T6). Anoten dónde se equivocó: ese es el primer candidato a caso adversarial.

In [ ]:
def cronometrar(f, *a, **k):
    t0 = time.time(); r = f(*a, **k); return r, round(time.time() - t0, 2)

for nombre, im in imgs.items():
    (crudo, js), seg = cronometrar(ver, im)
    print(f'[{nombre}] {seg}s')
    print('  JSON :', js)
    if js is None:
        print('  CRUDO:', crudo[:200], '   <- no parseo (cuenta como fallo, regla 2)')
    print()

# Anoten: la salida es JSON parseable? cuantos segundos? donde alucino un dato que no se ve?

## 3 · El enchufe: `sistema_multimodal(pregunta, imagen)` (T7 · Martín/Tomás)

Aquí entra el sistema de M3 de Amparo. Por ahora hay un **SUSTITUTO marcado** para que
el pipeline corra de punta a punta hoy. Martín/Tomás (T7) lo reemplazan por el M3 real
(`SistemaReglas`/`SistemaRAG`/`Agente` de `src/`), **misma firma**
`responder_m3(relato: str) -> str`.

> Para los 3.0 puntos la entrega final debe usar M3 real. Si no levanta en Colab, se deja
> el sustituto y se **declara en el README** (plan B del plan).

In [ ]:
# TODO T7 (Martin/Tomas): reemplazar este sustituto por el M3 real de Amparo.
#   Misma firma: responder_m3(relato: str) -> str
#   El M3 real vive en src/: SistemaReglas (brazo 4, el del motor de reglas) es el
#   candidato natural porque extrae hechos y decide con reglas.
def responder_m3(relato):
    # SUSTITUTO (marcado): no hay RAG ni agente aqui. Solo para que el pipeline corra hoy.
    return f'[SUSTITUTO M3] Relato recibido ({len(relato)} chars). Primeros 300: {relato[:300]}'

def sistema_multimodal(pregunta, imagen=None):
    """pregunta (+ imagen) -> respuesta de M3. Patron input: lo visto se suma al relato."""
    if imagen is None:
        return responder_m3(pregunta)
    crudo, js = ver(imagen)
    contexto = json.dumps(js, ensure_ascii=False) if js else crudo
    relato = pregunta + '\n\nDATOS DEL DOCUMENTO ADJUNTO:\n' + contexto
    return responder_m3(relato)

ejemplo = sistema_multimodal(PREGUNTA_USUARIO, list(imgs.values())[0])
print(ejemplo)

> 🔄 **Fin de lo de Santiago (T1 + T2).** Cuatro cosas ciertas: el setup corre desde
> cero · `ver()` corre sola sobre 3 imágenes · la latencia está anotada ·
> `sistema_multimodal` responde de punta a punta (con el sustituto).
>
> **Sigue:** Lucas (T3–T4) trae las imágenes reales y `casos_m4.csv`; Nico (T5–T6) el
> harness y la tabla de delta; Martín/Tomás (T7) el enchufe real de M3.